<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

# quick sanity check — should return a row count, not an error
cols = con.sql("""
    DESCRIBE SELECT * FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    ) LIMIT 1
""").df()
print(cols.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 1–2. Contract

1. **Unit of analysis**: one row = one piece of content, for one client, on one calendar day
   (grain confirmed: 0 duplicate client/content/date combos in month=2026-03).

2. **Table(s)**: `fact_content_daily_performance` — the daily-grain warehouse table.
   No other tables used yet.

3. **Time window**: developing on `month=2026-03` (a mid-panel month, full 31 days present).
   The `month=2026-06` (`_sample`) table is a sealed test month — used only to test query
   mechanics, never for label logic, since it's the natural outcome window for any
   past→future label.

4. **Label / proxy**: no pre-built label column exists in this table — every field is a raw
   GSC or GA4 metric. I construct a proxy: [pick one — see below], computed from GSC
   visibility and GA4 engagement columns.

5. **Excluded**: `ga4_data_available` / `gsc_data_available` are used only to filter rows
   (Section 3, availability query) — never as a model feature. Including them as a feature
   would let the model learn "tracked vs. untracked" instead of real performance, which is
   leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
before = con.sql("""
    SELECT COUNT(*) FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    ) WHERE month = '2026-03'
""").fetchone()[0]

after = con.sql("""
    SELECT COUNT(*) FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    ) WHERE month = '2026-03' AND ga4_data_available IS TRUE
""").fetchone()[0]

print(f"before: {before}, after: {after}, survived: {after/before:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

before: 9841378, after: 413966, survived: 4.2%


In [ ]:
grain_check = con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print(f"duplicate grain rows: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate grain rows: 0


In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

┌─────────┬────────────┬────────────┐
│ n_rows  │ start_date │  end_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Since gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions are consumed by the label, they're off-limits as features. Candidates from what's left:

gsc_clicks — same-day clicks. Knowable at the decision moment because it's logged the instant a search click happens, no lag.
sessions_ai — AI-referral session count. Knowable because AI-platform referral traffic is logged same-day like any other channel.
scroll_events — on-page scroll interactions. Knowable because it's a same-day behavioral log, independent of the engagement/session metrics used in the label.
sessions_organic — organic (non-AI, non-paid) session count. Knowable same-day as a channel-level session count, separate from the engaged-session figure used in the proxy.
ga4_total_engagement_sec — total engagement time in seconds. Knowable same-day, but flag: this is engagement-adjacent to the label. Worth discussing — do you want a cleaner 5th feature instead, like sessions_social or sessions_referral, to keep clean separation from the label?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.